In [ ]:
# ┌─────────────────────────────────────────┐
# │  Cell 1 · Environment Setup             │
# │  Verify GEMC, PyVista, VTK              │
# └─────────────────────────────────────────┘

import subprocess
import warnings
import pyvista as pv
pv.set_jupyter_backend('trame')

import vtk
vtk.vtkObject.GlobalWarningDisplayOff()  # suppress warnings

result = subprocess.run(["gemc", "--version"], capture_output=True, text=True)
print("GEMC version:", result.stdout.strip())


In [ ]:
# ┌─────────────────────────────────────────┐
# │  Cell 2 · Define Geometry               │
# │  Write and execute geometry script      │
# └─────────────────────────────────────────┘

target_py = """\
#!/usr/bin/env python3
from gconfiguration import autogeometry
from gvolume import GVolume

cfg = autogeometry("examples", "simple_flux")

world_size = 110
gvolume = GVolume("root")
gvolume.description = "World"
gvolume.make_box(world_size * 0.5, world_size * 0.5, world_size * 0.5)
gvolume.material = "G4_AIR"
gvolume.color = "ghostwhite"
gvolume.style = 0
gvolume.publish(cfg)

target_dz = 20
target_radius = 5
gvolume = GVolume("Target")
gvolume.mother = "root"
gvolume.description = "Simple Carbon Target"
gvolume.make_tube(0, target_radius, target_dz, 0, 360)
gvolume.material = "G4_C"
gvolume.color = "metallic, darkgreen"
gvolume.publish(cfg)

flux_z = 50
flux_dx = 1
flux_dim = world_size * 0.8
gvolume = GVolume("FluxPlane")
gvolume.mother = "root"
gvolume.description = "Flux Scoring Plane"
gvolume.make_box(flux_dim * 0.5, flux_dim * 0.5, flux_dx * 0.5)
gvolume.material = "G4_AIR"
gvolume.color = "FAFAD2"
gvolume.set_position(0, 0, flux_z)
gvolume.digitization = "flux"
gvolume.set_identifier("flux_plane", 1)
gvolume.publish(cfg)
"""

with open("target.py", "w") as f:
    f.write(target_py)
print("Geometry definitions written in target.py")

from run_geometry import run_geometry

run_geometry("target.py")


In [ ]:
# ┌─────────────────────────────────────────┐
# │  Cell 3 · Run 10,000 events in GEMC     │
# │  Load gsystem, use 2.2 GeV proton beam  │
# │  Write CSV output                       │
# └─────────────────────────────────────────┘

system='-gsystem="[{name: simple_flux}]"'
particle='-gparticle="[{name: proton, p: 2200}]"'
output='-gstreamer="[{format: csv, filename: out}]"'
nevents='-n=10000'
nthreads='-nthreads=1'

result = subprocess.run(["gemc", system, particle, output, nevents, nthreads], capture_output=True, text=True)
if result.returncode != 0:
    print("GEMC failed:")
    print(result.stderr)
else:
    print(result.stdout)

In [ ]:
# ┌─────────────────────────────────────────┐
# │  Cell 4 · Graph total energy deposited  │
# │  Use matplotlib and panda               │
# └─────────────────────────────────────────┘

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from io import StringIO

def plot_totEdep_histogram(df, bins=30, xlim=None):
    fig, ax = plt.subplots(figsize=(10, 6))

    pids = df['pid'].unique()
    colors = list(mcolors.TABLEAU_COLORS.values())

    # ── compute shared bin edges from the full dataset ───────
    data_range = df['totEdep']
    if xlim is not None:
        data_range = data_range.clip(*xlim)
    bin_edges = np.linspace(data_range.min(), data_range.max(), bins + 1)

    for i, pid in enumerate(sorted(pids)):
        subset = df[df['pid'] == pid]['totEdep']
        ax.hist(subset, bins=bin_edges, alpha=0.7,
                label=f'pid {pid}',
                color=colors[i % len(colors)],
                edgecolor='white', linewidth=0.5)

    if xlim is not None:
        ax.set_xlim(xlim)

    ax.set_xlabel('Total Energy Deposit (MeV)', fontsize=12)
    ax.set_ylabel('Counts', fontsize=12)
    ax.set_title('Total Energy Deposit by Particle ID', fontsize=14)
    ax.legend(title='Particle ID')
    ax.set_yscale('log')
    plt.tight_layout()
    plt.show()

# ── usage ────────────────────────────────────────────────────
df = pd.read_csv('out_t0_digitized.csv', sep=',', skipinitialspace=True)
plot_totEdep_histogram(df, xlim=(0.0, 0.01))